### Cell 10.01 — define project paths and frozen inputs

In [ ]:
# Cell 10.01
# Final QTL results synthesis notebook
# Load paths to all frozen analysis outputs.

from pathlib import Path
import pandas as pd
import numpy as np


def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()


MAP_FILE = (
    PROJECT_ROOT
    / "results"
    / "linkage_map"
    / "flyer_hartwig_structural_physical_map_final.xlsx"
)

QTL_FILE = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_all33_empirical_qtl_screen.xlsx"
)

PHYSICAL_QTL_FILE = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_qtl_physical_localization.xlsx"
)

SCN_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_scn_gm20_final_evidence_summary.xlsx"
)

LRN_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_lrn_gm02_final_candidates.xlsx"
)

PROT03_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_prot03_gm20_final_evidence_summary.xlsx"
)

DAYSFL_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_days_fl07_gm08_context.xlsx"
)


FROZEN_INPUTS = {
    "structural_map": MAP_FILE,
    "all33_qtl": QTL_FILE,
    "physical_qtl": PHYSICAL_QTL_FILE,
    "scn_candidates": SCN_FILE,
    "lrn_candidates": LRN_FILE,
    "prot03_candidates": PROT03_FILE,
    "days_fl_context": DAYSFL_FILE,
}


print("FROZEN INPUT INVENTORY")
print("=" * 90)

for name, path in FROZEN_INPUTS.items():
    print(
        f"{name:20s}",
        "EXISTS" if path.exists() else "MISSING",
        path
    )

### Cell 10.02 — audit workbook sheets and load core frozen tables

In [ ]:
# Cell 10.02
# Inspect frozen workbook structure and load the core synthesis tables.

for label, path in FROZEN_INPUTS.items():

    if path.exists():

        print()
        print(label.upper())
        print("-" * 90)

        try:
            xls = pd.ExcelFile(path)
            print(xls.sheet_names)

        except Exception as e:
            print("Could not inspect workbook:", e)


# Core QTL tables
all33 = pd.read_excel(
    QTL_FILE,
    sheet_name="all_33_empirical"
)

all_trait_peaks = pd.read_excel(
    QTL_FILE,
    sheet_name="all_trait_peaks"
)

suggestive_regions = pd.read_excel(
    QTL_FILE,
    sheet_name="suggestive_regions"
)


# Core structural map tables
ordered_framework = pd.read_excel(
    MAP_FILE,
    sheet_name="ordered_framework"
)

group_summary = pd.read_excel(
    MAP_FILE,
    sheet_name="group_summary"
)

physical_assignments = pd.read_excel(
    MAP_FILE,
    sheet_name="physical_assignments"
)

removed_joins = pd.read_excel(
    MAP_FILE,
    sheet_name="removed_joins"
)


print()
print("CORE TABLE SHAPES")
print("=" * 90)

print("all33:", all33.shape)
print("all_trait_peaks:", all_trait_peaks.shape)
print("suggestive_regions:", suggestive_regions.shape)
print("ordered_framework:", ordered_framework.shape)
print("group_summary:", group_summary.shape)
print("physical_assignments:", physical_assignments.shape)
print("removed_joins:", removed_joins.shape)

### Cell 10.03 — verify the final all-33-trait statistical summary

In [ ]:
# Cell 10.03
# Verify the final empirical QTL status across all 33 traits.

print("ALL-TRAIT EMPIRICAL QTL SUMMARY")
print("=" * 90)

print("Traits:", len(all33))

print()
print("Genome-wide status counts:")
print(
    all33[
        "genomewide_status"
    ]
    .value_counts(
        dropna=False
    )
)


suggestive = (
    all33
    .loc[
        all33["genomewide_status"]
        == "suggestive_10pct"
    ]
    .copy()
    .sort_values(
        "peak_empirical_p"
    )
    .reset_index(drop=True)
)


print()
print(
    "Suggestive loci:",
    len(suggestive)
)


display(
    suggestive[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "r2",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
)


# Explicit validation
expected_suggestive_traits = {
    "lrn",
    "scn_fi3",
    "prot_03",
    "days_fl_07"
}

observed_suggestive_traits = set(
    suggestive["trait"]
)


print()
print(
    "Expected suggestive traits recovered:",
    observed_suggestive_traits
    == expected_suggestive_traits
)


print(
    "No traits significant at 5%:",
    bool(
        (
            all33["lod"]
            <
            all33["lod_threshold_05pct"]
        ).all()
    )
)

### Cell 10.04 — construct the master four-locus synthesis table

In [ ]:
# Cell 10.04
# Build the authoritative synthesis table for the four suggestive loci.
#
# This table combines:
# statistical evidence,
# allele interpretation,
# physical resolution,
# and candidate-gene analysis status.

master_qtl = (
    suggestive[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "r2",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
    .copy()
)


manual_context = {

    "lrn": {
        "physical_assignment":
            "strong",

        "physical_resolution":
            "two_sided_anchor_bracket",

        "physical_region":
            "Gm02:20.165739-46.904614 Mb",

        "allele_interpretation":
            "Flyer allele increases LRN at peak",

        "candidate_analysis":
            "performed",

        "candidate_summary":
            (
                "8 evidence-aware candidates; "
                "Glyma.02G211800 strongest independent "
                "soybean auxin-pathway evidence"
            )
    },

    "scn_fi3": {
        "physical_assignment":
            "supported",

        "physical_resolution":
            "two_sided_anchor_bracket",

        "physical_region":
            "Gm20:28.303434-38.482498 Mb",

        "allele_interpretation":
            "Flyer allele reduces SCN female index",

        "candidate_analysis":
            "performed",

        "candidate_summary":
            (
                "9 high-priority genes; "
                "Glyma.20G104000 has direct SCN-responsive "
                "molecular evidence"
            )
    },

    "prot_03": {
        "physical_assignment":
            "supported",

        "physical_resolution":
            "exact_peak_anchor_only",

        "physical_region":
            "Satt440 = Gm20:49.904048 Mb",

        "allele_interpretation":
            "see frozen peak-effect table",

        "candidate_analysis":
            "descriptive_local_context",

        "candidate_summary":
            (
                "19 curated local functional candidates; "
                "no closed physical QTL interval"
            )
    },

    "days_fl_07": {
        "physical_assignment":
            "insufficient_single_anchor",

        "physical_resolution":
            "single_anchor_group_assignment",

        "physical_region":
            "Sat_162 = Gm08:8.327526 Mb; "
            "TMA2 is 2.742473 provisional cM away",

        "allele_interpretation":
            "Flyer allele associated with earlier flowering",

        "candidate_analysis":
            "not_performed",

        "candidate_summary":
            (
                "No candidate-gene window generated because "
                "TMA2 lacks a direct physical coordinate"
            )
    }
}


context_df = (
    pd.DataFrame
    .from_dict(
        manual_context,
        orient="index"
    )
    .reset_index()
    .rename(
        columns={
            "index": "trait"
        }
    )
)


master_qtl = (
    master_qtl
    .merge(
        context_df,
        on="trait",
        how="left",
        validate="one_to_one"
    )
)


# Sort by empirical P-value
master_qtl = (
    master_qtl
    .sort_values(
        "peak_empirical_p"
    )
    .reset_index(drop=True)
)


print("MASTER FOUR-QTL SYNTHESIS")
print("=" * 90)


display(master_qtl)


print()
print("Rows:", len(master_qtl))
print(
    "All context fields complete:",
    master_qtl[
        [
            "physical_assignment",
            "physical_resolution",
            "physical_region",
            "allele_interpretation",
            "candidate_analysis"
        ]
    ]
    .notna()
    .all()
    .all()
)

### Cell 10.05 — recover exact peak allele effects for all four suggestive loci

In [ ]:
# Cell 10.05
# Recover peak allele effects directly from frozen regional QTL sheets.
#
# Genotype coding:
#   0 = Hartwig
#   2 = Flyer
#
# effect_2_minus_0 = Flyer - Hartwig

REGION_SHEETS = {
    "lrn": "region_lrn",
    "scn_fi3": "region_scn_fi3",
    "prot_03": "region_prot_03",
    "days_fl_07": "region_days_fl_07",
}


effect_rows = []


for trait, sheet in REGION_SHEETS.items():

    region = pd.read_excel(
        QTL_FILE,
        sheet_name=sheet
    )

    peak_marker = (
        master_qtl
        .loc[
            master_qtl["trait"] == trait,
            "peak_marker"
        ]
        .iloc[0]
    )

    peak = (
        region
        .loc[
            region["marker"].astype(str)
            == str(peak_marker)
        ]
        .copy()
    )

    if len(peak) != 1:
        raise ValueError(
            f"{trait}: expected one peak row for "
            f"{peak_marker}, found {len(peak)}"
        )

    peak = peak.iloc[0]

    effect = float(
        peak["effect_2_minus_0"]
    )

    mean0 = float(
        peak["mean_genotype_0"]
    )

    mean2 = float(
        peak["mean_genotype_2"]
    )


    if trait == "scn_fi3":

        biological_direction = (
            "Flyer allele lowers female index"
            if effect < 0
            else
            "Hartwig allele lowers female index"
        )

    elif trait == "days_fl_07":

        biological_direction = (
            "Flyer allele associated with earlier flowering"
            if effect < 0
            else
            "Hartwig allele associated with earlier flowering"
        )

    else:

        biological_direction = (
            "Flyer allele increases trait"
            if effect > 0
            else
            "Flyer allele decreases trait"
        )


    effect_rows.append(
        {
            "trait": trait,
            "peak_marker": peak_marker,
            "mean_hartwig_0": mean0,
            "mean_flyer_2": mean2,
            "effect_flyer_minus_hartwig": effect,
            "biological_direction": biological_direction,
        }
    )


peak_effects = pd.DataFrame(
    effect_rows
)


print("PEAK ALLELE EFFECTS")
print("=" * 90)

display(peak_effects)


master_qtl = (
    master_qtl
    .drop(
        columns=[
            "allele_interpretation"
        ],
        errors="ignore"
    )
    .merge(
        peak_effects,
        on=[
            "trait",
            "peak_marker"
        ],
        how="left",
        validate="one_to_one"
    )
)


print()
print(
    "All four effects recovered:",
    master_qtl[
        "effect_flyer_minus_hartwig"
    ]
    .notna()
    .all()
)

### Cell 10.06 — audit and recover frozen physical-localization records

In [ ]:
# Cell 10.06
# Read every sheet from the frozen QTL physical-localization workbook
# and recover any trait-level records for the four suggestive loci.

physical_xls = pd.ExcelFile(
    PHYSICAL_QTL_FILE
)

print("PHYSICAL LOCALIZATION SHEETS")
print("=" * 90)
print(physical_xls.sheet_names)


physical_trait_records = []


for sheet in physical_xls.sheet_names:

    df = pd.read_excel(
        PHYSICAL_QTL_FILE,
        sheet_name=sheet
    )

    if "trait" not in df.columns:
        continue

    subset = (
        df
        .loc[
            df["trait"].isin(
                master_qtl["trait"]
            )
        ]
        .copy()
    )

    if len(subset) > 0:

        subset[
            "_source_sheet"
        ] = sheet

        physical_trait_records.append(
            subset
        )


if physical_trait_records:

    physical_trait_records = (
        pd.concat(
            physical_trait_records,
            ignore_index=True,
            sort=False
        )
    )

else:

    physical_trait_records = pd.DataFrame()


print()
print(
    "Recovered physical-localization rows:",
    len(physical_trait_records)
)

print(
    "Columns:"
)

print(
    physical_trait_records.columns.tolist()
)


display(
    physical_trait_records
)

### Cell 10.07 — summarize candidate-evidence depth for each suggestive locus

In [ ]:
# Cell 10.07
# Recover candidate-analysis counts from the four frozen downstream workbooks.

def read_first_existing_sheet(
    workbook,
    preferred_sheets
):

    xls = pd.ExcelFile(workbook)

    for sheet in preferred_sheets:

        if sheet in xls.sheet_names:

            return pd.read_excel(
                workbook,
                sheet_name=sheet
            ), sheet

    return pd.DataFrame(), None


# ---------------------------------------------------------
# SCN
# ---------------------------------------------------------

scn_high, scn_sheet = (
    read_first_existing_sheet(
        SCN_FILE,
        [
            "high_priority_9",
            "high_priority",
            "Tier_1"
        ]
    )
)


# ---------------------------------------------------------
# LRN
# ---------------------------------------------------------

lrn_short, lrn_sheet = (
    read_first_existing_sheet(
        LRN_FILE,
        [
            "evidence_shortlist",
            "shortlist"
        ]
    )
)


# ---------------------------------------------------------
# prot_03
# ---------------------------------------------------------

prot_short, prot_sheet = (
    read_first_existing_sheet(
        PROT03_FILE,
        [
            "curated_candidates",
            "focused_candidates"
        ]
    )
)


# ---------------------------------------------------------
# days_fl_07
# ---------------------------------------------------------

days_summary, days_sheet = (
    read_first_existing_sheet(
        DAYSFL_FILE,
        [
            "locus_summary"
        ]
    )
)


candidate_evidence_summary = pd.DataFrame(
    [
        {
            "trait": "scn_fi3",
            "candidate_analysis_level":
                "prioritized_candidate_region",
            "candidate_count":
                len(scn_high),
            "candidate_sheet":
                scn_sheet,
            "top_evidence":
                (
                    "Glyma.20G104000: direct "
                    "SCN-responsive molecular evidence"
                )
        },

        {
            "trait": "lrn",
            "candidate_analysis_level":
                "evidence_aware_shortlist",
            "candidate_count":
                len(lrn_short),
            "candidate_sheet":
                lrn_sheet,
            "top_evidence":
                (
                    "Glyma.02G211800: independent "
                    "soybean auxin-receptor evidence"
                )
        },

        {
            "trait": "prot_03",
            "candidate_analysis_level":
                "descriptive_anchor_context",
            "candidate_count":
                len(prot_short),
            "candidate_sheet":
                prot_sheet,
            "top_evidence":
                (
                    "Glyma.20G228900: supported "
                    "cysteine-biosynthesis candidate"
                )
        },

        {
            "trait": "days_fl_07",
            "candidate_analysis_level":
                "not_performed",
            "candidate_count":
                0,
            "candidate_sheet":
                days_sheet,
            "top_evidence":
                (
                    "No physical candidate window; "
                    "single-anchor chromosome assignment only"
                )
        }
    ]
)


print("CANDIDATE-EVIDENCE SUMMARY")
print("=" * 90)

display(candidate_evidence_summary)


print()
print("Recovered source sheets:")
print("SCN:", scn_sheet)
print("LRN:", lrn_sheet)
print("prot_03:", prot_sheet)
print("days_fl_07:", days_sheet)

### Cell 10.08 — create the first manuscript-ready four-QTL table

In [ ]:
# Cell 10.08
# Merge candidate evidence and create a compact manuscript-ready table.

master_qtl = (
    master_qtl
    .merge(
        candidate_evidence_summary,
        on="trait",
        how="left",
        validate="one_to_one"
    )
)


manuscript_qtl_table = (
    master_qtl[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "lod",
            "r2",
            "peak_empirical_p",
            "genomewide_status",
            "effect_flyer_minus_hartwig",
            "biological_direction",
            "physical_assignment",
            "physical_resolution",
            "physical_region",
            "candidate_analysis_level",
            "candidate_count",
            "top_evidence"
        ]
    ]
    .copy()
)


# Friendly manuscript column names
manuscript_qtl_table = (
    manuscript_qtl_table
    .rename(
        columns={
            "trait": "Trait",
            "peak_marker": "Peak_marker",
            "structural_group": "Linkage_fragment",
            "candidate_chr": "Physical_chr",
            "n": "N",
            "lod": "LOD",
            "r2": "R2",
            "peak_empirical_p": "Empirical_P",
            "genomewide_status": "Genomewide_status",
            "effect_flyer_minus_hartwig":
                "Flyer_minus_Hartwig_effect",
            "biological_direction":
                "Allele_effect_interpretation",
            "physical_assignment":
                "Physical_assignment",
            "physical_resolution":
                "Physical_resolution",
            "physical_region":
                "Physical_context",
            "candidate_analysis_level":
                "Candidate_analysis",
            "candidate_count":
                "Candidate_count",
            "top_evidence":
                "Highest_priority_evidence"
        }
    )
)


# Round display statistics only.
manuscript_qtl_table[
    "LOD"
] = manuscript_qtl_table[
    "LOD"
].round(3)

manuscript_qtl_table[
    "R2"
] = manuscript_qtl_table[
    "R2"
].round(3)

manuscript_qtl_table[
    "Empirical_P"
] = manuscript_qtl_table[
    "Empirical_P"
].round(4)

manuscript_qtl_table[
    "Flyer_minus_Hartwig_effect"
] = manuscript_qtl_table[
    "Flyer_minus_Hartwig_effect"
].round(3)


print("MANUSCRIPT-READY SUGGESTIVE-QTL TABLE")
print("=" * 90)

display(
    manuscript_qtl_table
)


print()
print(
    "Rows:",
    len(manuscript_qtl_table)
)

print(
    "Missing values in key fields:",
    manuscript_qtl_table[
        [
            "LOD",
            "Empirical_P",
            "Flyer_minus_Hartwig_effect",
            "Physical_resolution",
            "Candidate_analysis"
        ]
    ]
    .isna()
    .sum()
    .sum()
)

### Cell 10.09 — establish output folders and save current synthesis tables

In [ ]:
# Cell 10.09
# Establish permanent Notebook 10 output locations.
# From this point onward, every manuscript table/figure should be saved.

TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


MASTER_QTL_CSV = (
    TABLE_DIR
    / "flyer_hartwig_four_suggestive_qtl_master.csv"
)

MANUSCRIPT_QTL_CSV = (
    TABLE_DIR
    / "flyer_hartwig_manuscript_suggestive_qtl_table.csv"
)

PEAK_EFFECTS_CSV = (
    TABLE_DIR
    / "flyer_hartwig_suggestive_qtl_peak_effects.csv"
)

CANDIDATE_SUMMARY_CSV = (
    TABLE_DIR
    / "flyer_hartwig_suggestive_qtl_candidate_evidence.csv"
)


# Save full-precision analytical tables
master_qtl.to_csv(
    MASTER_QTL_CSV,
    index=False
)

peak_effects.to_csv(
    PEAK_EFFECTS_CSV,
    index=False
)

candidate_evidence_summary.to_csv(
    CANDIDATE_SUMMARY_CSV,
    index=False
)


# Save manuscript-formatted table
manuscript_qtl_table.to_csv(
    MANUSCRIPT_QTL_CSV,
    index=False
)


print("NOTEBOOK 10 TABLE OUTPUTS")
print("=" * 100)

for path in [
    MASTER_QTL_CSV,
    MANUSCRIPT_QTL_CSV,
    PEAK_EFFECTS_CSV,
    CANDIDATE_SUMMARY_CSV,
]:
    print(
        f"{'SAVED' if path.exists() else 'FAILED':8s}",
        path
    )

### Cell 10.10 — build and save the final all-33-trait QTL table
* This becomes the authoritative supplementary statistical table.

In [ ]:
# Cell 10.10
# Build the manuscript/supplementary table for all 33 phenotype scans.
#
# IMPORTANT:
# Statistical status comes only from empirical genome-wide permutation testing.

all33_manuscript = all33.copy()


# Add a simple significance-order field.
status_order = {
    "suggestive_10pct": 1,
    "not_genomewide_significant": 2,
}

all33_manuscript[
    "_status_order"
] = (
    all33_manuscript[
        "genomewide_status"
    ]
    .map(status_order)
    .fillna(99)
)


all33_manuscript = (
    all33_manuscript
    .sort_values(
        [
            "_status_order",
            "peak_empirical_p",
            "lod"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .drop(
        columns="_status_order"
    )
    .reset_index(drop=True)
)


ALL33_CSV = (
    TABLE_DIR
    / "flyer_hartwig_all33_empirical_qtl_results.csv"
)

all33_manuscript.to_csv(
    ALL33_CSV,
    index=False
)


print("ALL-33-TRAIT QTL TABLE")
print("=" * 100)

print("Rows:", len(all33_manuscript))

print()
print("Status counts:")
print(
    all33_manuscript[
        "genomewide_status"
    ]
    .value_counts(dropna=False)
)

print()
print(
    "Traits exceeding 5% genome-wide threshold:",
    (
        all33_manuscript["lod"]
        >=
        all33_manuscript["lod_threshold_05pct"]
    ).sum()
)

print()
print("Saved:")
print(ALL33_CSV)


display(
    all33_manuscript[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "r2",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
)

### Cell 10.11 — build and save final structural-map summary
* This creates a manuscript-ready map overview from the frozen structural map rather than manually retyping the 22 fragments.

In [ ]:
# Cell 10.11
# Create manuscript-ready structural-map summary.
#
# No map reconstruction occurs here.
# Everything is derived from the frozen final structural workbook.

map_summary_final = (
    group_summary
    .copy()
)


print("GROUP SUMMARY COLUMNS")
print("=" * 100)
print(map_summary_final.columns.tolist())


# ----------------------------------------------------------
# Global frozen-map statistics
# ----------------------------------------------------------

n_framework_markers = len(
    ordered_framework
)

n_fragments = len(
    group_summary
)

n_removed_joins = len(
    removed_joins
)


# Detect likely interval-count and map-length columns safely.
print()
print("FINAL STRUCTURAL MAP")
print("=" * 100)

print(
    f"Ordered framework markers: {n_framework_markers}"
)

print(
    f"Structural fragments:      {n_fragments}"
)

print(
    f"Removed joins:             {n_removed_joins}"
)


# ----------------------------------------------------------
# Save complete fragment-level summary
# ----------------------------------------------------------

MAP_SUMMARY_CSV = (
    TABLE_DIR
    / "flyer_hartwig_final_structural_map_summary.csv"
)

map_summary_final.to_csv(
    MAP_SUMMARY_CSV,
    index=False
)


REMOVED_JOINS_CSV = (
    TABLE_DIR
    / "flyer_hartwig_final_removed_structural_joins.csv"
)

removed_joins.to_csv(
    REMOVED_JOINS_CSV,
    index=False
)


PHYSICAL_ASSIGNMENTS_CSV = (
    TABLE_DIR
    / "flyer_hartwig_final_physical_assignments.csv"
)

physical_assignments.to_csv(
    PHYSICAL_ASSIGNMENTS_CSV,
    index=False
)


print()
print("Saved:")
print(MAP_SUMMARY_CSV)
print(REMOVED_JOINS_CSV)
print(PHYSICAL_ASSIGNMENTS_CSV)


display(map_summary_final)

### Cell 10.12 — export one consolidated Notebook 10 workbook
* This is the important checkpoint. It bundles the synthesis tables into one Excel workbook while preserving the individual CSVs.

In [ ]:
# Cell 10.12
# Export consolidated Notebook 10 synthesis workbook.

SYNTHESIS_XLSX = (
    TABLE_DIR
    / "flyer_hartwig_final_qtl_results_synthesis.xlsx"
)


with pd.ExcelWriter(
    SYNTHESIS_XLSX,
    engine="openpyxl"
) as writer:

    # Main manuscript-oriented QTL table
    manuscript_qtl_table.to_excel(
        writer,
        sheet_name="suggestive_qtl_main",
        index=False
    )

    # Full-precision four-locus analytical table
    master_qtl.to_excel(
        writer,
        sheet_name="suggestive_qtl_full",
        index=False
    )

    # All 33 empirical scans
    all33_manuscript.to_excel(
        writer,
        sheet_name="all_33_traits",
        index=False
    )

    # Peak allele effects
    peak_effects.to_excel(
        writer,
        sheet_name="peak_allele_effects",
        index=False
    )

    # Candidate evidence
    candidate_evidence_summary.to_excel(
        writer,
        sheet_name="candidate_evidence",
        index=False
    )

    # Final structural map
    map_summary_final.to_excel(
        writer,
        sheet_name="map_fragments",
        index=False
    )

    # Physical chromosome assignments
    physical_assignments.to_excel(
        writer,
        sheet_name="physical_assignments",
        index=False
    )

    # Removed false/chaining joins
    removed_joins.to_excel(
        writer,
        sheet_name="removed_joins",
        index=False
    )


# ----------------------------------------------------------
# Verify workbook
# ----------------------------------------------------------

check_xls = pd.ExcelFile(
    SYNTHESIS_XLSX
)


print("FINAL NOTEBOOK 10 SYNTHESIS WORKBOOK")
print("=" * 100)

print("Saved:")
print(SYNTHESIS_XLSX)

print()
print("Exists:", SYNTHESIS_XLSX.exists())

if SYNTHESIS_XLSX.exists():
    print(
        "Size:",
        f"{SYNTHESIS_XLSX.stat().st_size / 1024:.1f} KB"
    )

print()
print("Sheets:")

for sheet in check_xls.sheet_names:
    print(" -", sheet)


print()
print(
    "Workbook sheet count:",
    len(check_xls.sheet_names)
)

### Cell 10.13 — ranked peak LOD across all 33 traits

In [ ]:
# Cell 10.13
# Figure: peak LOD scores across all 33 traits.
# Saves:
#   - source CSV
#   - PNG
#   - PDF

import matplotlib.pyplot as plt


fig10_13_data = (
    all33_manuscript[
        [
            "trait",
            "lod",
            "lod_threshold_10pct",
            "lod_threshold_05pct",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
    .copy()
    .sort_values(
        "lod",
        ascending=True
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Save figure source table
# ----------------------------------------------------------

FIG10_13_CSV = (
    TABLE_DIR
    / "figure10_13_all33_peak_lod_source.csv"
)

fig10_13_data.to_csv(
    FIG10_13_CSV,
    index=False
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.5, 11)
)

y = np.arange(
    len(fig10_13_data)
)

ax.barh(
    y,
    fig10_13_data["lod"]
)

ax.set_yticks(y)

ax.set_yticklabels(
    fig10_13_data["trait"],
    fontsize=8
)

ax.set_xlabel(
    "Peak single-marker LOD score"
)

ax.set_ylabel(
    "Trait"
)

ax.set_title(
    "Peak QTL signal across 33 traits"
)

ax.grid(
    axis="x",
    alpha=0.25
)


# Mark the four empirical 10%-suggestive loci
for i, row in fig10_13_data.iterrows():

    if row["genomewide_status"] == "suggestive_10pct":

        ax.text(
            row["lod"] + 0.04,
            i,
            "*",
            va="center",
            fontsize=11
        )


ax.text(
    0.99,
    0.01,
    "* empirical genome-wide suggestive at 10%",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=8
)


plt.tight_layout()


FIG10_13_PNG = (
    FIGURE_DIR
    / "figure10_13_all33_peak_lod.png"
)

FIG10_13_PDF = (
    FIGURE_DIR
    / "figure10_13_all33_peak_lod.pdf"
)


fig.savefig(
    FIG10_13_PNG,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIG10_13_PDF,
    bbox_inches="tight"
)

plt.show()

plt.close(fig)


print("FIGURE 10.13 SAVED")
print("=" * 90)

print(FIG10_13_CSV)
print(FIG10_13_PNG)
print(FIG10_13_PDF)

### Cell 10.14 — empirical P values for the four suggestive loci
* This makes the statistical status visually explicit: all four are below 0.10 but above 0.05.

In [ ]:
# Cell 10.14 — REVISED
# Figure: empirical genome-wide P values for the four suggestive loci.
#
# Improvements:
#   - legend moved completely above plotting region
#   - exact empirical P values labeled above bars
#   - 0.05 and 0.10 reference lines remain unobstructed
#   - source table + PNG + PDF saved
#   - existing Figure 10.14 files are overwritten

import matplotlib.pyplot as plt
import numpy as np


# ----------------------------------------------------------
# Prepare source data
# ----------------------------------------------------------

fig10_14_data = (
    master_qtl[
        [
            "trait",
            "peak_marker",
            "candidate_chr",
            "lod",
            "peak_empirical_p",
            "genomewide_status"
        ]
    ]
    .copy()
    .sort_values(
        "peak_empirical_p",
        ascending=True
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Save / overwrite source table
# ----------------------------------------------------------

FIG10_14_CSV = (
    TABLE_DIR
    / "figure10_14_four_qtl_empirical_p_source.csv"
)

fig10_14_data.to_csv(
    FIG10_14_CSV,
    index=False
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.5, 6.2)
)


labels = (
    fig10_14_data["trait"].astype(str)
    + "\n"
    + fig10_14_data["peak_marker"].astype(str)
)


x = np.arange(
    len(fig10_14_data)
)


bars = ax.bar(
    x,
    fig10_14_data["peak_empirical_p"],
    width=0.72
)


# ----------------------------------------------------------
# Genome-wide reference levels
# ----------------------------------------------------------

ax.axhline(
    0.10,
    linestyle="--",
    linewidth=1.3,
    label="10% genome-wide level"
)

ax.axhline(
    0.05,
    linestyle=":",
    linewidth=1.3,
    label="5% genome-wide level"
)


# ----------------------------------------------------------
# Exact P-value labels above bars
# ----------------------------------------------------------

for bar, pval in zip(
    bars,
    fig10_14_data["peak_empirical_p"]
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{pval:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )


# ----------------------------------------------------------
# Axes
# ----------------------------------------------------------

ax.set_xticks(x)

ax.set_xticklabels(
    labels,
    fontsize=9
)

ax.set_ylabel(
    "Empirical genome-wide P value"
)

ax.set_xlabel(
    "Trait and peak marker"
)

ax.set_title(
    "Empirical support for the four suggestive QTL",
    pad=38
)


# Leave visual space above 0.10 line.
ax.set_ylim(
    0,
    0.115
)


ax.set_yticks(
    np.arange(
        0,
        0.111,
        0.02
    )
)


ax.grid(
    axis="y",
    alpha=0.25
)


# ----------------------------------------------------------
# Legend ABOVE plot — avoids overlap with 0.10 line
# ----------------------------------------------------------

ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.01),
    ncol=2,
    frameon=False,
    fontsize=9
)


# Explicitly reserve top margin for title + legend.
fig.subplots_adjust(
    top=0.82,
    bottom=0.16,
    left=0.12,
    right=0.97
)


# ----------------------------------------------------------
# Save / overwrite figure
# ----------------------------------------------------------

FIG10_14_PNG = (
    FIGURE_DIR
    / "figure10_14_four_qtl_empirical_p.png"
)

FIG10_14_PDF = (
    FIGURE_DIR
    / "figure10_14_four_qtl_empirical_p.pdf"
)


fig.savefig(
    FIG10_14_PNG,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIG10_14_PDF,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


# ----------------------------------------------------------
# Verify saved outputs
# ----------------------------------------------------------

print("REVISED FIGURE 10.14 SAVED")
print("=" * 90)

for path in [
    FIG10_14_CSV,
    FIG10_14_PNG,
    FIG10_14_PDF
]:
    print(
        f"{'SAVED' if path.exists() else 'FAILED':8s}",
        path
    )

### Cell 10.15 — regional genetic profiles for all four suggestive loci
* This creates four separate figures, one per locus.
* No subplots, and no physical coordinates are invented.

In [ ]:
# Cell 10.15 — REVISED
# Regional genetic profiles for the four empirical 10%-suggestive loci.
#
# Improvements:
#   - adds the trait-specific empirical 10% genome-wide LOD threshold
#   - labels the threshold directly on the plot
#   - preserves peak-marker annotation
#   - dynamically expands y-axis so labels do not overlap
#   - saves each source table + PNG + PDF
#
# IMPORTANT:
# The horizontal threshold is trait-specific and comes from the
# empirical permutation test for that phenotype.

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


REGION_SHEETS = {
    "lrn": "region_lrn",
    "scn_fi3": "region_scn_fi3",
    "prot_03": "region_prot_03",
    "days_fl_07": "region_days_fl_07",
}


def detect_cm_column(df):

    candidates = [
        "kosambi_cm_structural_provisional",
        "kosambi_cm_provisional",
        "kosambi_cm",
        "position_cm",
        "cm"
    ]

    for col in candidates:

        if col in df.columns:
            return col

    raise KeyError(
        "Could not identify genetic-position column. "
        f"Available columns: {df.columns.tolist()}"
    )


regional_figure_records = []


for trait, sheet in REGION_SHEETS.items():

    # ------------------------------------------------------
    # Load regional QTL profile
    # ------------------------------------------------------

    region = pd.read_excel(
        QTL_FILE,
        sheet_name=sheet
    ).copy()

    cm_col = detect_cm_column(
        region
    )

    region = (
        region
        .sort_values(cm_col)
        .reset_index(drop=True)
    )


    # ------------------------------------------------------
    # Retrieve frozen trait-level QTL information
    # ------------------------------------------------------

    qtl_row = (
        master_qtl
        .loc[
            master_qtl["trait"] == trait
        ]
        .copy()
    )

    if len(qtl_row) != 1:

        raise ValueError(
            f"{trait}: expected one master_qtl row, "
            f"found {len(qtl_row)}"
        )

    qtl_row = qtl_row.iloc[0]

    peak_marker = qtl_row[
        "peak_marker"
    ]

    lod_threshold_10 = float(
        qtl_row[
            "lod_threshold_10pct"
        ]
    )

    peak_lod_master = float(
        qtl_row[
            "lod"
        ]
    )


    # ------------------------------------------------------
    # Identify peak row in regional profile
    # ------------------------------------------------------

    peak_row = (
        region
        .loc[
            region["marker"].astype(str)
            == str(peak_marker)
        ]
    )

    if len(peak_row) != 1:

        raise ValueError(
            f"{trait}: expected exactly one peak row "
            f"for {peak_marker}; found {len(peak_row)}"
        )

    peak_row = peak_row.iloc[0]


    # ------------------------------------------------------
    # Add threshold information to saved source data
    # ------------------------------------------------------

    region[
        "empirical_lod_threshold_10pct"
    ] = lod_threshold_10

    region[
        "genomewide_status"
    ] = qtl_row[
        "genomewide_status"
    ]


    # ------------------------------------------------------
    # Save source table
    # ------------------------------------------------------

    source_csv = (
        TABLE_DIR
        / f"figure10_15_{trait}_regional_profile_source.csv"
    )

    region.to_csv(
        source_csv,
        index=False
    )


    # ------------------------------------------------------
    # Plot
    # ------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(8.5, 5.5)
    )


    # Regional LOD profile
    ax.plot(
        region[cm_col],
        region["lod"],
        marker="o",
        linewidth=1.6,
        markersize=5
    )


    # Peak marker
    ax.scatter(
        [peak_row[cm_col]],
        [peak_row["lod"]],
        s=85,
        zorder=5
    )


    # ------------------------------------------------------
    # Empirical 10% genome-wide LOD threshold
    # ------------------------------------------------------

    ax.axhline(
        y=lod_threshold_10,
        linestyle="--",
        linewidth=1.4,
        label=(
            f"10% empirical genome-wide threshold "
            f"(LOD = {lod_threshold_10:.2f})"
        )
    )


    # ------------------------------------------------------
    # Peak label
    # ------------------------------------------------------

    ax.annotate(
        f"{peak_marker}\nLOD = {peak_row['lod']:.2f}",
        (
            peak_row[cm_col],
            peak_row["lod"]
        ),
        xytext=(7, 10),
        textcoords="offset points",
        fontsize=9,
        ha="left",
        va="bottom"
    )


    # ------------------------------------------------------
    # Other marker labels
    # ------------------------------------------------------

    if len(region) <= 12:

        for _, row in region.iterrows():

            if str(row["marker"]) == str(peak_marker):
                continue

            ax.annotate(
                str(row["marker"]),
                (
                    row[cm_col],
                    row["lod"]
                ),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=7,
                alpha=0.80
            )


    # ------------------------------------------------------
    # Axes
    # ------------------------------------------------------

    ax.set_xlabel(
        "Provisional Kosambi position (cM)"
    )

    ax.set_ylabel(
        "Single-marker LOD"
    )

    ax.set_title(
        f"{trait}: regional QTL profile"
    )


    # Dynamic y-axis:
    # ensure room above the peak and threshold label.
    y_max = max(
        region["lod"].max(),
        lod_threshold_10
    )

    y_margin = max(
        0.25,
        y_max * 0.12
    )

    ax.set_ylim(
        0,
        y_max + y_margin
    )


    ax.grid(
        alpha=0.25
    )


    # Legend placed away from threshold/peak
    ax.legend(
        loc="upper left",
        frameon=False,
        fontsize=8
    )


    plt.tight_layout()


    # ------------------------------------------------------
    # Save figure
    # ------------------------------------------------------

    png_file = (
        FIGURE_DIR
        / f"figure10_15_{trait}_regional_lod.png"
    )

    pdf_file = (
        FIGURE_DIR
        / f"figure10_15_{trait}_regional_lod.pdf"
    )


    fig.savefig(
        png_file,
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        pdf_file,
        bbox_inches="tight"
    )


    plt.show()
    plt.close(fig)


    # ------------------------------------------------------
    # Record figure manifest
    # ------------------------------------------------------

    regional_figure_records.append(
        {
            "trait": trait,
            "sheet": sheet,
            "peak_marker": peak_marker,
            "peak_lod": peak_lod_master,
            "empirical_lod_threshold_10pct":
                lod_threshold_10,
            "lod_above_threshold":
                peak_lod_master - lod_threshold_10,
            "n_markers": len(region),
            "cm_column": cm_col,
            "source_csv": str(source_csv),
            "png_file": str(png_file),
            "pdf_file": str(pdf_file)
        }
    )


# ----------------------------------------------------------
# Save updated regional figure manifest
# ----------------------------------------------------------

regional_figure_manifest = pd.DataFrame(
    regional_figure_records
)


REGIONAL_MANIFEST_CSV = (
    TABLE_DIR
    / "figure10_15_regional_profiles_manifest.csv"
)

regional_figure_manifest.to_csv(
    REGIONAL_MANIFEST_CSV,
    index=False
)


print("REVISED REGIONAL QTL FIGURES")
print("=" * 100)

display(
    regional_figure_manifest[
        [
            "trait",
            "peak_marker",
            "peak_lod",
            "empirical_lod_threshold_10pct",
            "lod_above_threshold",
            "n_markers"
        ]
    ]
)


print()
print("Manifest saved:")
print(REGIONAL_MANIFEST_CSV)

### Cell 10.16 — final 22-fragment structural-map length figure
* This produces a compact overview of the structurally corrected map.

In [ ]:
# Cell 10.16
# Figure: provisional Kosambi map length of the 22 final structural fragments.
#
# Saves source CSV + PNG + PDF.

print("AVAILABLE GROUP-SUMMARY COLUMNS")
print("=" * 90)
print(group_summary.columns.tolist())


def find_first_column(
    df,
    candidates
):

    for col in candidates:

        if col in df.columns:
            return col

    return None


group_col = find_first_column(
    group_summary,
    [
        "structural_group",
        "group",
        "linkage_group",
        "fragment",
        "group_name"
    ]
)


length_col = find_first_column(
    group_summary,
    [
        "kosambi_length_provisional",
        "kosambi_cm_provisional",
        "total_kosambi_cm",
        "kosambi_length_cm",
        "map_length_kosambi",
        "length_kosambi_cm"
    ]
)


marker_count_col = find_first_column(
    group_summary,
    [
        "n_markers",
        "marker_count",
        "n_framework_markers",
        "framework_markers"
    ]
)


if group_col is None:

    raise KeyError(
        "Could not identify structural-group column."
    )


# If the summary workbook does not contain a length column,
# calculate fragment span directly from ordered_framework.

if length_col is not None:

    keep_cols = [
        group_col,
        length_col
    ]

    if marker_count_col is not None:
        keep_cols.append(
            marker_count_col
        )

    fig10_16_data = (
        group_summary[
            keep_cols
        ]
        .copy()
    )

    fig10_16_data = fig10_16_data.rename(
        columns={
            group_col: "structural_group",
            length_col: "provisional_kosambi_cm"
        }
    )

    if marker_count_col is not None:

        fig10_16_data = fig10_16_data.rename(
            columns={
                marker_count_col: "n_markers"
            }
        )


else:

    framework_group_col = find_first_column(
        ordered_framework,
        [
            "structural_group",
            "group",
            "linkage_group",
            "fragment"
        ]
    )

    framework_cm_col = find_first_column(
        ordered_framework,
        [
            "kosambi_cm_structural_provisional",
            "kosambi_cm_provisional",
            "kosambi_cm"
        ]
    )


    if (
        framework_group_col is None
        or framework_cm_col is None
    ):

        raise KeyError(
            "Could not reconstruct fragment lengths "
            "from ordered_framework."
        )


    fig10_16_data = (
        ordered_framework
        .groupby(
            framework_group_col,
            as_index=False
        )
        .agg(
            provisional_kosambi_cm=(
                framework_cm_col,
                lambda x: x.max() - x.min()
            ),
            n_markers=(
                framework_cm_col,
                "size"
            )
        )
        .rename(
            columns={
                framework_group_col:
                    "structural_group"
            }
        )
    )


fig10_16_data = (
    fig10_16_data
    .sort_values(
        "provisional_kosambi_cm",
        ascending=True
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------
# Validation
# ----------------------------------------------------------

print()
print("STRUCTURAL MAP FIGURE DATA")
print("=" * 90)

print(
    "Fragments:",
    len(fig10_16_data)
)

print(
    "Total provisional Kosambi length:",
    fig10_16_data[
        "provisional_kosambi_cm"
    ].sum()
)


display(
    fig10_16_data
)


# ----------------------------------------------------------
# Save source table
# ----------------------------------------------------------

FIG10_16_CSV = (
    TABLE_DIR
    / "figure10_16_structural_fragment_lengths_source.csv"
)

fig10_16_data.to_csv(
    FIG10_16_CSV,
    index=False
)


# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8, 8)
)


y = np.arange(
    len(fig10_16_data)
)


ax.barh(
    y,
    fig10_16_data[
        "provisional_kosambi_cm"
    ]
)


ax.set_yticks(y)

ax.set_yticklabels(
    fig10_16_data[
        "structural_group"
    ],
    fontsize=8
)


ax.set_xlabel(
    "Provisional Kosambi map length (cM)"
)

ax.set_ylabel(
    "Structural linkage fragment"
)

ax.set_title(
    "Final structurally corrected Flyer × Hartwig linkage map"
)

ax.grid(
    axis="x",
    alpha=0.25
)


plt.tight_layout()


FIG10_16_PNG = (
    FIGURE_DIR
    / "figure10_16_structural_fragment_lengths.png"
)

FIG10_16_PDF = (
    FIGURE_DIR
    / "figure10_16_structural_fragment_lengths.pdf"
)


fig.savefig(
    FIG10_16_PNG,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    FIG10_16_PDF,
    bbox_inches="tight"
)

plt.show()

plt.close(fig)


print()
print("FIGURE 10.16 SAVED")
print("=" * 90)

print(FIG10_16_CSV)
print(FIG10_16_PNG)
print(FIG10_16_PDF)

### Cell 10.17 — build and save manuscript Table 1
* This is the compact main-text table for the four suggestive loci.

In [ ]:
# Cell 10.17
# Build publication-oriented main-text Table 1.
# Saves CSV + Excel.

table1 = (
    master_qtl[
        [
            "trait",
            "peak_marker",
            "structural_group",
            "candidate_chr",
            "n",
            "lod",
            "r2",
            "peak_empirical_p",
            "effect_flyer_minus_hartwig",
            "biological_direction",
            "physical_resolution",
            "physical_region"
        ]
    ]
    .copy()
)


table1 = table1.rename(
    columns={
        "trait": "Trait",
        "peak_marker": "Peak marker",
        "structural_group": "Linkage fragment",
        "candidate_chr": "Chromosome",
        "n": "N",
        "lod": "LOD",
        "r2": "R²",
        "peak_empirical_p": "Empirical P",
        "effect_flyer_minus_hartwig": "Flyer − Hartwig effect",
        "biological_direction": "Allelic interpretation",
        "physical_resolution": "Physical resolution",
        "physical_region": "Physical context"
    }
)


# manuscript-friendly rounding
table1["LOD"] = table1["LOD"].round(2)
table1["R²"] = table1["R²"].round(3)
table1["Empirical P"] = table1["Empirical P"].round(3)
table1["Flyer − Hartwig effect"] = (
    table1["Flyer − Hartwig effect"].round(3)
)


TABLE1_CSV = (
    TABLE_DIR
    / "table1_four_suggestive_qtl.csv"
)

TABLE1_XLSX = (
    TABLE_DIR
    / "table1_four_suggestive_qtl.xlsx"
)


table1.to_csv(
    TABLE1_CSV,
    index=False
)

table1.to_excel(
    TABLE1_XLSX,
    index=False
)


print("TABLE 1 SAVED")
print("=" * 90)

print(TABLE1_CSV)
print(TABLE1_XLSX)

display(table1)

### Cell 10.18 — build supplementary-table index
* This organizes the important outputs for the manuscript and supplement.

In [ ]:
# Cell 10.18
# Build and save a manuscript/supplement table index.

supplement_index = pd.DataFrame(
    [
        {
            "table_id": "Table 1",
            "description":
                "Four empirical 10%-suggestive QTL",
            "file":
                str(TABLE1_XLSX)
        },
        {
            "table_id": "Table S1",
            "description":
                "All 33 empirical QTL scan results",
            "file":
                str(ALL33_CSV)
        },
        {
            "table_id": "Table S2",
            "description":
                "Final structural linkage-map fragment summary",
            "file":
                str(MAP_SUMMARY_CSV)
        },
        {
            "table_id": "Table S3",
            "description":
                "Final physical chromosome assignments",
            "file":
                str(PHYSICAL_ASSIGNMENTS_CSV)
        },
        {
            "table_id": "Table S4",
            "description":
                "Removed structurally unsupported joins",
            "file":
                str(REMOVED_JOINS_CSV)
        },
        {
            "table_id": "Table S5",
            "description":
                "Peak allele effects for four suggestive loci",
            "file":
                str(PEAK_EFFECTS_CSV)
        },
        {
            "table_id": "Table S6",
            "description":
                "Candidate-evidence summary for suggestive loci",
            "file":
                str(CANDIDATE_SUMMARY_CSV)
        }
    ]
)


SUPPLEMENT_INDEX_CSV = (
    TABLE_DIR
    / "manuscript_table_index.csv"
)

supplement_index.to_csv(
    SUPPLEMENT_INDEX_CSV,
    index=False
)


print("MANUSCRIPT TABLE INDEX")
print("=" * 90)

display(supplement_index)

print()
print("Saved:")
print(SUPPLEMENT_INDEX_CSV)

### Cell 10.19 — create complete figure/table manifest
* This gives us one audit trail of all Notebook 10 outputs created so far.

In [ ]:
# Cell 10.19
# Create and save complete Notebook 10 output manifest.

from pathlib import Path


manifest_records = []


# ----------------------------------------------------------
# Tables
# ----------------------------------------------------------

for path in sorted(TABLE_DIR.glob("*")):

    if path.is_file():

        manifest_records.append(
            {
                "category": "table",
                "name": path.name,
                "path": str(path),
                "size_kb": round(
                    path.stat().st_size / 1024,
                    2
                )
            }
        )


# ----------------------------------------------------------
# Figures
# ----------------------------------------------------------

for path in sorted(FIGURE_DIR.glob("*")):

    if path.is_file():

        manifest_records.append(
            {
                "category": "figure",
                "name": path.name,
                "path": str(path),
                "size_kb": round(
                    path.stat().st_size / 1024,
                    2
                )
            }
        )


output_manifest = pd.DataFrame(
    manifest_records
)


OUTPUT_MANIFEST_CSV = (
    TABLE_DIR
    / "notebook10_output_manifest.csv"
)

output_manifest.to_csv(
    OUTPUT_MANIFEST_CSV,
    index=False
)


print("NOTEBOOK 10 OUTPUT MANIFEST")
print("=" * 90)

print(
    output_manifest[
        "category"
    ]
    .value_counts()
)

print()

display(output_manifest)

print()
print("Saved:")
print(OUTPUT_MANIFEST_CSV)

### Cell 10.20 — final synthesis workbook + validation checkpoint
* This makes one final frozen Notebook 10 workbook containing the essential synthesis outputs.

In [ ]:
# Cell 10.20
# Final Notebook 10 checkpoint workbook.
# This is the file to preserve when Notebook 10 is frozen.

FINAL_NOTEBOOK10_XLSX = (
    TABLE_DIR
    / "flyer_hartwig_notebook10_final_synthesis.xlsx"
)


with pd.ExcelWriter(
    FINAL_NOTEBOOK10_XLSX,
    engine="openpyxl"
) as writer:

    table1.to_excel(
        writer,
        sheet_name="Table1_main_qtl",
        index=False
    )

    all33_manuscript.to_excel(
        writer,
        sheet_name="TableS1_all33_qtl",
        index=False
    )

    master_qtl.to_excel(
        writer,
        sheet_name="four_qtl_full",
        index=False
    )

    peak_effects.to_excel(
        writer,
        sheet_name="peak_allele_effects",
        index=False
    )

    candidate_evidence_summary.to_excel(
        writer,
        sheet_name="candidate_evidence",
        index=False
    )

    map_summary_final.to_excel(
        writer,
        sheet_name="map_fragments",
        index=False
    )

    physical_assignments.to_excel(
        writer,
        sheet_name="physical_assignments",
        index=False
    )

    removed_joins.to_excel(
        writer,
        sheet_name="removed_joins",
        index=False
    )

    supplement_index.to_excel(
        writer,
        sheet_name="table_index",
        index=False
    )

    output_manifest.to_excel(
        writer,
        sheet_name="output_manifest",
        index=False
    )


# ----------------------------------------------------------
# Final validation summary
# ----------------------------------------------------------

validation = {
    "n_traits_scanned": len(all33_manuscript),

    "n_suggestive_10pct": int(
        (
            all33_manuscript[
                "genomewide_status"
            ]
            == "suggestive_10pct"
        ).sum()
    ),

    "n_significant_05pct": int(
        (
            all33_manuscript["lod"]
            >=
            all33_manuscript[
                "lod_threshold_05pct"
            ]
        ).sum()
    ),

    "n_structural_fragments":
        len(group_summary),

    "n_ordered_framework_markers":
        len(ordered_framework),

    "n_removed_structural_joins":
        len(removed_joins),

    "final_notebook10_file_exists":
        FINAL_NOTEBOOK10_XLSX.exists()
}


validation_df = pd.DataFrame(
    validation.items(),
    columns=[
        "metric",
        "value"
    ]
)


VALIDATION_CSV = (
    TABLE_DIR
    / "notebook10_final_validation.csv"
)

validation_df.to_csv(
    VALIDATION_CSV,
    index=False
)


print("NOTEBOOK 10 FINAL VALIDATION")
print("=" * 90)

display(validation_df)

print()
print("FINAL CHECKPOINT:")
print(FINAL_NOTEBOOK10_XLSX)

print()
print(
    "Exists:",
    FINAL_NOTEBOOK10_XLSX.exists()
)

if FINAL_NOTEBOOK10_XLSX.exists():

    print(
        "Size:",
        f"{FINAL_NOTEBOOK10_XLSX.stat().st_size / 1024:.1f} KB"
    )

print()
print("Validation file:")
print(VALIDATION_CSV)